In [3]:
import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI()


# -----------------------------
# Hilfsfunktionen
# -----------------------------
DOMAINS = ["A", "B", "C", "D", "E"]

OUTPUT_CONTRACT_DEFAULT = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "severity": <int or null>,
  "confidence": <float 0..1 or null>,
  "findings": <list of strings>
}

Rules:
- findings must be short factual phrases grounded in the call text.
- if no findings are available, return [].
- do NOT invent findings.
""".strip()



def next_free_run_path(results_dir: Path, model: str, temperature: float) -> Path:
    """
    Erzeugt:
      run_7_gpt-5.2_t1.csv
    mit automatisch nächster Run-Nummer.
    """

    results_dir.mkdir(parents=True, exist_ok=True)

    # vorhandene run_X_*.csv suchen
    nums = []
    for p in results_dir.glob("run_*.csv"):
        m = re.search(r"run_(\d+)_", p.name)
        if m:
            nums.append(int(m.group(1)))

    n = (max(nums) + 1) if nums else 1

    # Model-Name dateisystem-sicher machen
    safe_model = re.sub(r"[^\w\.-]+", "-", model)

    # Temperature hübsch formatieren
    if float(temperature).is_integer():
        temp_str = f"t{int(temperature)}"
    else:
        temp_str = f"t{str(temperature).replace('.', '_')}"

    filename = f"run_{n}_{safe_model}_{temp_str}.csv"
    return results_dir / filename


def _extract_json_from_response_obj(resp_body: dict) -> dict:
    out = resp_body.get("output", [])
    texts = []
    for item in out:
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text") and "text" in c:
                texts.append(c["text"])
    if not texts and resp_body.get("output_text"):
        texts = [resp_body["output_text"]]
    if not texts:
        raise ValueError("Kein output_text gefunden")

    text = "\n".join(texts).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError(f"Kein JSON im Text: {text[:200]}...")
    return json.loads(m.group(0))


def parse_batch_output_jsonl_text_to_df(jsonl_text: str) -> pd.DataFrame:
    by_call = {}

    for line in jsonl_text.splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)

        custom_id = rec.get("custom_id")  # Notruf_12__A
        if not custom_id or "__" not in custom_id:
            continue

        call_id, domain = custom_id.split("__", 1)
        domain = domain.upper()
        if domain not in DOMAINS:
            continue

        by_call.setdefault(call_id, {"file": call_id})

        err = rec.get("error")
        if err:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = json.dumps(err, ensure_ascii=False)
            continue

        body = (rec.get("response") or {}).get("body") or {}
        try:
            data = _extract_json_from_response_obj(body)
            sev = data.get("severity")
            conf = data.get("confidence")
            findings = data.get("findings", [])
            if findings is None:
                findings = []
            if not isinstance(findings, list):
                findings = [str(findings)]
        except Exception as e:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = f"parse_error: {e}"
            continue

        by_call[call_id][f"severity_{domain}"] = sev
        by_call[call_id][f"confidence_{domain}"] = conf
        by_call[call_id][f"findings_{domain}"] = json.dumps(findings, ensure_ascii=False)
        by_call[call_id][f"error_{domain}"] = None

    df = pd.DataFrame(list(by_call.values()))
    df["call_nr"] = df["file"].str.extract(r"(\d+)").astype("Int64")
    df = df.sort_values(["call_nr", "file"], na_position="last").reset_index(drop=True)

    cols = ["file", "call_nr"]
    for d in DOMAINS:
        cols += [f"severity_{d}", f"confidence_{d}", f"findings_{d}", f"error_{d}"]
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]


def _download_file_text(client: OpenAI, file_id: str) -> str:
    """
    Robust: lädt die Datei-Inhalte als Text.
    (SDK-Details können variieren; das deckt die üblichen Fälle ab.)
    """
    resp = client.files.content(file_id)
    # je nach SDK: resp kann bytes, ein HTTP Response-Objekt mit .read(), oder schon text sein
    if hasattr(resp, "read"):
        data = resp.read()
    else:
        data = resp
    if isinstance(data, bytes):
        return data.decode("utf-8", errors="replace")
    return str(data)


# -----------------------------
# Hauptfunktion
# -----------------------------
def run_abcde_batch_experiment(
    model: str,
    temperature: float,
    *,
    calls_dir: Path,
    guidelines_dir: Path,
    results_dir: Path,
    completion_window: str = "24h",
    output_contract: str = OUTPUT_CONTRACT_DEFAULT,
    wait: bool = True,
    poll_every_seconds: int = 20,
    max_wait_seconds: int = 60 * 60,  # 1h default; bei completion_window=24h ggf. erhöhen oder wait=False
):
    """
    Führt ein komplettes ABCDE-Batch-Experiment aus und speichert CSV als nächste freie run_X.csv.

    Returns:
      dict mit batch_id, output_jsonl_path, csv_path, df (falls fertig), status
    """

    # OpenAI Client
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENAI_API_KEY fehlt (env).")
    client = OpenAI()

    calls_dir = Path(calls_dir).resolve()
    guidelines_dir = Path(guidelines_dir).resolve()
    results_dir = Path(results_dir).resolve()

    assert calls_dir.exists(), f"calls_dir nicht gefunden: {calls_dir}"
    assert guidelines_dir.exists(), f"guidelines_dir nicht gefunden: {guidelines_dir}"

    call_files = sorted(list(calls_dir.glob("*.md")) + list(calls_dir.glob("*.txt")))
    assert call_files, f"Keine Notrufe gefunden in {calls_dir}"

    # Guidelines laden
    DOMAIN_FILES = {d: guidelines_dir / f"{d.lower()}_guidelines.md" for d in DOMAINS}
    domain_guidelines = {}
    for d, p in DOMAIN_FILES.items():
        if not p.exists():
            raise FileNotFoundError(f"Guideline fehlt: {p}")
        domain_guidelines[d] = p.read_text(encoding="utf-8").strip()

    # Batch input schreiben (immer in results_dir ablegen)
    results_dir.mkdir(parents=True, exist_ok=True)
    batch_input_path = results_dir / "batch_input_abcde.jsonl"

    with batch_input_path.open("w", encoding="utf-8") as f:
        for fp in call_files:
            transcript = fp.read_text(encoding="utf-8").strip()

            for domain in DOMAINS:
                req = {
                    "custom_id": f"{fp.stem}__{domain}",
                    "method": "POST",
                    "url": "/v1/responses",
                    "body": {
                        "model": model,
                        #"temperature": temperature,  #Bei 5 mini
                        "input": [
                            {"role": "system", "content": [{"type": "input_text", "text": domain_guidelines[domain]}]},
                            {"role": "system", "content": [{"type": "input_text", "text": output_contract}]},
                            {"role": "user",   "content": [{"type": "input_text", "text": transcript}]},
                        ],
                        "text": {"format": {"type": "json_object"}},
                    },
                }
                f.write(json.dumps(req, ensure_ascii=False) + "\n")

    # Upload + Batch create
    upload = client.files.create(file=batch_input_path.open("rb"), purpose="batch")
    batch = client.batches.create(
        input_file_id=upload.id,
        endpoint="/v1/responses",
        completion_window=completion_window,
    )

    # Wenn nicht warten: sofort zurückgeben (batch_id zum späteren retrieve)
    if not wait:
        return {
            "status": batch.status,
            "batch_id": batch.id,
            "batch_input_path": str(batch_input_path),
            "output_jsonl_path": None,
            "csv_path": None,
            "df": None,
        }

    # Polling
    t0 = time.time()
    while True:
        b = client.batches.retrieve(batch.id)
        if b.status in ("completed", "failed", "cancelled", "expired"):
            break
        if max_wait_seconds is not None and (time.time() - t0) > max_wait_seconds:
            return {
                "status": b.status,
                "batch_id": batch.id,
                "batch_input_path": str(batch_input_path),
                "output_jsonl_path": None,
                "csv_path": None,
                "df": None,
                "note": f"Nicht fertig nach max_wait_seconds={max_wait_seconds}. Setze wait=False oder erhöhe max_wait_seconds.",
            }
        time.sleep(poll_every_seconds)

    if b.status != "completed":
        # error_file_id ggf. herunterladen zur Diagnose
        err_text = None
        if getattr(b, "error_file_id", None):
            err_text = _download_file_text(client, b.error_file_id)
        return {
            "status": b.status,
            "batch_id": batch.id,
            "batch_input_path": str(batch_input_path),
            "output_jsonl_path": None,
            "csv_path": None,
            "df": None,
            "error_file_text": err_text,
        }

    # Output JSONL downloaden und lokal speichern
    if not b.output_file_id:
        raise RuntimeError("Batch completed, aber output_file_id fehlt.")
    output_text = _download_file_text(client, b.output_file_id)

    output_jsonl_path = results_dir / f"batch_output_{batch.id}.jsonl"
    output_jsonl_path.write_text(output_text, encoding="utf-8")

    # JSONL -> DF -> CSV
    df = parse_batch_output_jsonl_text_to_df(output_text)
    csv_path = next_free_run_path(results_dir, model=model, temperature=temperature)    
    df.to_csv(csv_path, index=False, encoding="utf-8")

    return {
        "status": b.status,
        "batch_id": batch.id,
        "batch_input_path": str(batch_input_path),
        "output_jsonl_path": str(output_jsonl_path),
        "csv_path": str(csv_path),
        "df": df,
    }

In [13]:
from pathlib import Path

res = run_abcde_batch_experiment(
    model="gpt-4o",
    temperature=1,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    completion_window="24h",
    wait=True,                 # setzt auf False, wenn du nicht pollen willst
    max_wait_seconds=None,     # 1h; bei 24h window ggf. erhöhen oder wait=False
)

print(res["status"], res["csv_path"])
df = res["df"]
df.head()

KeyboardInterrupt: 

In [9]:
print(res)

{'status': 'validating', 'batch_id': 'batch_69a1a1ca68dc8190a850b245c3ede1de', 'batch_input_path': '/Users/s.franke/Development/master_clean/experiments/experiment_3/results/batch_input_abcde.jsonl', 'output_jsonl_path': None, 'csv_path': None, 'df': None}


In [5]:
batch_id = "batch_69a1ae5c2be481908e05e9c4503773b4"

In [6]:


b = client.batches.retrieve(batch_id)

print("Status:", b.status)
print("Error file id:", b.error_file_id)
print("Output file id:", b.output_file_id)

Status: completed
Error file id: None
Output file id: file-TsirzmiXHUQfv5mieoAXo9


In [7]:
### Hier muss noch der Batch mit der obrigen ID geladen und ausgewertet werden! 

output_text = _download_file_text(client, b.output_file_id)

#output_jsonl_path = results_dir / f"batch_output_{batch.id}.jsonl"
#output_jsonl_path.write_text(output_text, encoding="utf-8")

# JSONL -> DF -> CSV
#df = parse_batch_output_jsonl_text_to_df(output_text)
#csv_path = next_free_run_path(results_dir, model=model, temperature=temperature)    
#df.to_csv(csv_path, index=False, encoding="utf-8")

In [10]:
#output_jsonl_path = "results/batch_output_7.jsonl"
#output_jsonl_path.write_text(output_text, encoding="utf-8")

df = parse_batch_output_jsonl_text_to_df(output_text)
#csv_path = next_free_run_path(, model=model, temperature=temperature)   
df.to_csv("results/run_7.csv", index=False, encoding="utf-8")